# PolicyPal — three RAG approaches

Traditional RAG · PageIndex · Agentic RAG over the OmniCorp handbook.

Set a provider key in `.env` (`OPENROUTER_API_KEY` recommended) for real answers, or
leave it blank to run the offline **mock** provider — it quotes real source text so the
mechanical verifier still passes.

In [1]:
from policypal import PolicyPal
from policypal.config import load_config

cfg = load_config()
print('provider :', cfg.provider, '| model:', getattr(cfg, cfg.provider + '_model', 'n/a') if cfg.provider != 'mock' else 'mock')
print('embeddings:', cfg.embedding_provider, '(dim', cfg.embedding_dim, ')')

provider : openrouter | model: openai/gpt-5.4-nano
embeddings: openrouter (dim 1536 )


In [2]:
pal = PolicyPal('OmniCorp_Handbook_Challenge V2.pdf', cfg)
print(f'{len(pal.blocks)} blocks -> {len(pal.chunks)} chunks, {len(pal.node_index)} tree nodes')

376 blocks -> 27 chunks, 17 tree nodes


## Inspect the inferred outline (§1.1 design note)
Eyeball this against the PDF's table of contents before trusting chunking/tree build.

In [3]:
from policypal.parse import print_outline
print_outline(pal.blocks)

p  1  sz 26.0  EMPLOYEE HANDBOOK
p  1  sz 16.0  Policies & Procedures v2024.1
p  2  sz 14.0  1. Remote Work & Flexibility
p  3  sz 14.0  2. Workplace Harassment Prevention
p  4  sz 14.0  3. Social Media Usage Guidelines
p  5  sz 14.0  4. Corporate Travel & Expense Reporting
p  6  sz 14.0  5. Intellectual Property Rights
p  7  sz 14.0  6. Conflict of Interest
p  8  sz 14.0  7. Data Privacy & GDPR Compliance
p  9  sz 14.0  8. Dress Code & Office Etiquette
p 10  sz 14.0  9. Emergency Evacuation Procedures
p 11  sz 14.0  10. Performance Review Cycle
p 12  sz 14.0  11. Mental Health & Wellbeing Resources
p 13  sz 14.0  APPENDIX B: Holiday Schedule
p 14  sz 14.0  URGENT AMENDMENT: Remote Work Update (June 2024)
p 14  sz 14.0  IT Security: Password Complexity


## The PageIndex tree (TOC the reasoning-search navigates)

In [4]:
from policypal.tree import render_toc
print(render_toc(pal.tree))

n0001 | EMPLOYEE HANDBOOK | EMPLOYEE HANDBOOK | pp.1-14
  n0002 | Policies & Procedures v2024.1 | This document contains the operating principles of OmniCorp. Please read carefully. Generated by HR Dept. Page 1 OmniCorp Analytics CONFIDENTIAL - DO NOT DISTRI… | pp.1-14
    n0003 | 1. Remote Work & Flexibility | OmniCorp believes in a balanced lifestyle. Standard Policy: Employees are eligible to work remotely up to 3 days per week (Tue/Wed/Thu). Mondays and Fridays are… | pp.2-3
    n0004 | 2. Workplace Harassment Prevention | The organization is committed to maintaining a workplace environment that is free from harassment, intimidation, and retaliation, and is dedicated to promoting … | pp.3-4
    n0005 | 3. Social Media Usage Guidelines | As a representative of our organization, it is imperative that all employees adhere to the Social Media Usage Guidelines outlined in this policy section. The pu… | pp.4-5
    n0006 | 4. Corporate Travel & Expense Reporting | As part of our organizat

## Ask an answerable question through all three approaches

Each claim is marked `✅` only if its quoted span was **mechanically re-located** in the
source PDF by [`policypal/verify.py`](policypal/verify.py) — the check is code, not another
LLM, so it can't hallucinate. The quote, section, and page are printed so an HR specialist
can confirm the answer at a glance. (A claim whose quote can't be found is shown `⚠️`.)

In [5]:
question = 'What are the password length requirements?'

for approach, ans in pal.answer_all(question).items():
    print('=' * 72)
    print('APPROACH:', approach)
    print('-' * 72)
    print(ans.render())
    print()

APPROACH: traditional
------------------------------------------------------------------------
✅ All passwords must be exactly 14 characters long; passwords shorter or longer than 14 characters will be rejected.
     ↳ [EMPLOYEE HANDBOOK > Policies & Procedures v2024.1 > IT Security: Password Complexity, p.14] “To ensure security, all passwords must be exactly 14 characters long. Passwords shorter or longer than 14
characters will be rejected.”
— tokens: 1,563 in + 82 out  ·  1 LLM call(s)  ·  est. cost $0.0004

APPROACH: pageindex
------------------------------------------------------------------------
✅ Passwords must be exactly 14 characters long; passwords shorter or longer will be rejected.
     ↳ [EMPLOYEE HANDBOOK > Policies & Procedures v2024.1 > IT Security: Password Complexity, p.14] “To ensure security, all passwords must be exactly 14 characters long. Passwords shorter or longer than 14
characters will be rejected.”
— tokens: 1,112 in + 104 out  ·  2 LLM call(s)  ·  est. co

## Two trust-critical behaviors: a superseded policy, and an unanswerable question

These are the cases that normally erode HR's trust. `answer_routed` auto-picks an approach
(and records which one); `render()` still shows the verified quote + page behind every claim.

1. **Superseded policy.** Section 1 says employees may work remotely *3 days/week*, but the
   **June-2024 URGENT AMENDMENT** overrides it to *1 day/week (Fridays)*. A correct answer
   must reflect the amendment, not the stale section. *(Reasoning happens in the configured
   model; the offline mock quotes a real span but does not reason about supersession.)*
2. **Not in the handbook.** There is no pet policy, so the system should **abstain** rather
   than invent one. Abstention needs a reasoning model — see `run_eval.ipynb` for the full
   unanswerable-question set.

In [6]:
edge_cases = [
    'How many days per week can I currently work remotely?',   # superseded by the June-2024 amendment
    'Can I bring my dog to the office?',                        # not in the handbook -> should abstain
]

for q in edge_cases:
    ans = pal.answer_routed(q)                 # auto-routes; records the approach actually used
    print('=' * 72)
    print('Q:', q, '   (approach:', ans.retrieval.get('approach', '?'), ')')
    print('-' * 72)
    print(ans.render())
    print()

Q: How many days per week can I currently work remotely?    (approach: pageindex )
------------------------------------------------------------------------
✅ Currently, remote work is limited to 1 day per week maximum, and that day must be a Friday.
     ↳ [EMPLOYEE HANDBOOK > Policies & Procedures v2024.1 > URGENT AMENDMENT: Remote Work Update (June 2024), p.14] “Due to real estate utilization requirements, remote work is now limited to 1 day per week maximum. This day must be a Friday. All other days require in-person attendance.”
— tokens: 1,717 in + 132 out  ·  2 LLM call(s)  ·  est. cost $0.0005

Q: Can I bring my dog to the office?    (approach: traditional (fallback from pageindex) )
------------------------------------------------------------------------
ABSTAINED — the handbook does not appear to contain this answer.
— tokens: 1,669 in + 14 out  ·  1 LLM call(s)  ·  est. cost $0.0004



## The audit trail
Each approach records *how* it retrieved — chunk ids, the PageIndex reasoning, or the
agent's full tool trace.

In [7]:
import json
ans = pal.answer(question, approach='agentic')
print(json.dumps(ans.retrieval, indent=2, default=str))

{
  "approach": "agentic",
  "trace": [
    {
      "tool": "browse_toc",
      "input": {}
    },
    {
      "tool": "read_section",
      "input": {
        "node_id": "n0016"
      }
    },
    {
      "tool": "verify_quote",
      "input": {
        "quote": "all passwords must be exactly 14 characters long. Passwords shorter or longer than 14 characters will be rejected.",
        "source_id": "n0016"
      }
    },
    {
      "tool": "submit_answer",
      "input": {
        "claims": [
          {
            "text": "Passwords must be exactly 14 characters long; passwords shorter or longer than 14 characters will be rejected.",
            "citations": [
              {
                "source_id": "n0016",
                "quote": "all passwords must be exactly 14 characters long. Passwords shorter or longer than 14\ncharacters will be rejected."
              }
            ]
          }
        ]
      }
    }
  ]
}
